In [ ]:
from ipywidgets import Dropdown, Textarea, Button, VBox, HBox, Layout, HTML
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import time

# --- NLLB Model and Translation Logic ---

NLLB_LANGUAGES = {
    "English": "eng_Latn",
    "Tamil":   "tam_Taml",
    "Hindi":   "hin_Deva",
    "Yoruba":  "yor_Latn",
    "Igbo":    "ibo_Latn",
    "Nepali":  "nep_Deva",
    "Sinhala": "sin_Sinh",
    "Telugu":  "tel_Telu"
}
LANGUAGE_NAMES = list(NLLB_LANGUAGES.keys())

model = None
tokenizer = None
MODEL_NAME = "facebook/nllb-200-distilled-600M"

def initialize_translator():
    """Initializes the NLLB model and tokenizer."""
    global model, tokenizer
    if model is None or tokenizer is None:
        print(f"Loading {MODEL_NAME}...")
        start_time = time.time()
        try:
            tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
            model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
            # Use GPU if available
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model.to(device)
            print(f"✅ Model and tokenizer loaded on {device} in {time.time() - start_time:.2f} seconds.")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            model = None
            tokenizer = None

def translate_text_nllb(text, src_lang_code, tgt_lang_code):
    """Translates text using AutoModel and AutoTokenizer with correct NLLB language tags."""
    global model, tokenizer
    if model is None or tokenizer is None:
        return "ERROR: Model not loaded. Please wait or run initialization again."
    try:
        device = model.device
        # Prepare inputs and target prefix language token
        inputs = tokenizer(text, return_tensors="pt").to(device)

        # Generate translation with forced target language token
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang_code),
            max_length=256
        )

        # Decode output tokens
        translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
        return translation
    except Exception as e:
        return f"Translation failed: {e}"

# --- ipywidgets GUI ---

# Titles
title_html = HTML(
    value="<h2 style='color:#2E86C1; text-align:center;'>🌍 NLLB Multi-Language Translator</h2>"
)

# Language Dropdowns
src_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='English',
    description='Source:',
    layout=Layout(width='45%')
)

tgt_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='Tamil',
    description='Target:',
    layout=Layout(width='45%')
)

swap_button = Button(description='🔁 Swap', layout=Layout(width='10%', height='35px'))

def swap_languages_event(b):
    src_lang_dropdown.value, tgt_lang_dropdown.value = tgt_lang_dropdown.value, src_lang_dropdown.value

swap_button.on_click(swap_languages_event)

# Input Label and Textarea
input_label = HTML("<b style='color:#1F618D;'>Input Text:</b>")
input_textarea = Textarea(
    placeholder='Enter text to translate...',
    layout=Layout(width='90%', height='100px')
)

# Output Label and Textarea
output_label = HTML("<b style='color:#117864;'>Translated Output:</b>")
output_textarea = Textarea(
    placeholder='Translated text will appear here...',
    disabled=True,
    layout=Layout(width='90%', height='100px')
)

# Translate Button
translate_button = Button(
    description='🚀 TRANSLATE',
    button_style='info',
    tooltip='Click to translate text',
    layout=Layout(width='200px', height='40px')
)

status_label = HTML("<i style='color:gray;'>Ready to translate...</i>")

def translate_button_event(b):
    input_text = input_textarea.value.strip()
    if not input_text:
        status_label.value = "<i style='color:red;'>⚠ Please enter text to translate.</i>"
        return

    src_lang = src_lang_dropdown.value
    tgt_lang = tgt_lang_dropdown.value
    src_code = NLLB_LANGUAGES.get(src_lang)
    tgt_code = NLLB_LANGUAGES.get(tgt_lang)

    if not src_code or not tgt_code:
        status_label.value = "<i style='color:red;'>⚠ Invalid language selection.</i>"
        return

    translate_button.description = "⏳ Translating..."
    translate_button.disabled = True
    status_label.value = f"<i style='color:gray;'>Translating from {src_lang} → {tgt_lang}...</i>"

    translated_text = translate_text_nllb(input_text, src_code, tgt_code)

    output_textarea.disabled = False
    output_textarea.value = translated_text
    output_textarea.disabled = True

    translate_button.description = "🚀 TRANSLATE"
    translate_button.disabled = False
    status_label.value = "<i style='color:green;'>✅ Translation complete!</i>"

translate_button.on_click(translate_button_event)

# --- Layout ---
language_box = HBox([src_lang_dropdown, swap_button, tgt_lang_dropdown],
                    layout=Layout(justify_content='space-between', margin='10px 0'))

button_box = HBox([translate_button],
                  layout=Layout(justify_content='center', margin='10px 0'))

app_layout = VBox([
    title_html,
    language_box,
    input_label, input_textarea,
    button_box,
    output_label, output_textarea,
    status_label
], layout=Layout(padding='15px', border='2px solid #D6DBDF', border_radius='10px', width='100%'))

print("Displaying the translator interface...")
display(app_layout)

# Initialize the model
initialize_translator()

Displaying the translator interface...


Loading facebook/nllb-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model and tokenizer loaded on cuda in 81.35 seconds.
